In [2]:
import os
import sys
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from google import genai
from transformers import pipeline


class MusicEmotionPipeline:
    def __init__(self, author="Dineshkumar"):
        self.author = author
        self.raw_df = None
        self.cleaned_df = None
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()

        # Standard GEMS framework categories used in the Emotify dataset
        self.emotion_cols = [
            'amazement', 'solemnity', 'tenderness', 'nostalgia',
            'calmness', 'power', 'joyful_activation', 'tension', 'sadness'
        ]

    def load_kaggle_dataset(self):
        """Downloads and loads the Emotify dataset via KaggleHub."""
        print(">> Ingesting data via KaggleHub...")
        try:
            path = kagglehub.dataset_download("yash9439/emotify-emotion-classificaiton-in-songs")

            # Resolve directory structure to find the CSV
            csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
            if not csv_files:
                for root, dirs, files in os.walk(path):
                    csv_files = [os.path.join(root, f) for f in files if f.endswith('.csv')]
                    if csv_files:
                        full_path = csv_files[0]
                        break
                else:
                    raise FileNotFoundError("Could not find any CSV files in the downloaded dataset.")
            else:
                full_path = os.path.join(path, csv_files[0])

            self.raw_df = pd.read_csv(full_path)
            print(f">> Dataset loaded successfully. Shape: {self.raw_df.shape[0]} rows, {self.raw_df.shape[1]} columns.")
            return True

        except Exception as e:
            print(f"!! Failed to load data: {e}. Falling back to synthetic baseline.")
            self._generate_fallback_data()
            return False

    def _generate_fallback_data(self):
        """Backup dataset generator to prevent pipeline crashes during offline testing."""
        np.random.seed(42)
        genres = ['rock', 'classical', 'pop', 'electronic']
        mock_data = {
            'track_id': range(1, 401),
            'genre': np.random.choice(genres, 400),
            'liked': np.random.choice([0, 1], 400, p=[0.4, 0.6]),
            'disliked': np.random.choice([0, 1], 400, p=[0.7, 0.3]),
            'age': np.random.randint(18, 60, size=400)
        }
        for emotion in self.emotion_cols:
            mock_data[emotion] = np.random.choice([0, 1], 400, p=[0.7, 0.3])
        self.raw_df = pd.DataFrame(mock_data)

    def preprocess_and_engineer_features(self):
        """Cleans headers, imputes missing values, and builds metrics."""
        print(">> Running preprocessing and feature engineering...")
        df = self.raw_df.copy()

        # Clean up column white-spaces to avoid indexing errors
        df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

        # Simple median imputation for numerical tracking data
        num_cols = df.select_dtypes(include=[np.number]).columns
        imputer = SimpleImputer(strategy='median')
        df[num_cols] = imputer.fit_transform(df[num_cols])

        # Feature Engineering: Sum active emotions for a density scale
        active_emotions = [c for c in self.emotion_cols if c in df.columns]
        df['emotional_density'] = df[active_emotions].sum(axis=1)

        # Target assignment based on the highest scoring emotion profile
        if active_emotions:
            df['primary_emotion'] = df[active_emotions].idxmax(axis=1)
        else:
            df['primary_emotion'] = 'neutral'

        df['encoded_emotion'] = self.label_encoder.fit_transform(df['primary_emotion'])

        # Plot distribution cleanly without deprecated seaborn parameters
        plt.figure(figsize=(10, 5))
        sns.countplot(data=df, x='genre', hue='genre', palette='viridis', legend=False)
        plt.title(f'Genre Distribution - Project Lead: {self.author}')
        plt.tight_layout()
        plt.savefig('genre_distribution.png')
        plt.close()

        self.cleaned_df = df
        print(">> Preprocessing completed. Visualizations cached locally.")

    def cluster_audio_signatures(self):

        print(">> Starting unsupervised segmentation...")
        features = [c for c in self.emotion_cols if c in self.cleaned_df.columns]
        X_cluster = self.cleaned_df[features]

        # KMeans Segmentation
        kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
        self.cleaned_df['kmeans_cluster'] = kmeans.fit_predict(X_cluster)
        print(f">> KMeans execution complete. Cluster layout:\n{self.cleaned_df['kmeans_cluster'].value_counts()}")

        # Dendrogram computation for validation
        Z = linkage(X_cluster.head(30), method='ward')
        plt.figure(figsize=(10, 4))
        dendrogram(Z)
        plt.title('Hierarchical Clustering Tree (Sample Baseline)')
        plt.tight_layout()
        plt.savefig('hierarchical_dendrogram.png')
        plt.close()
        print(">> Unsupervised validation map saved as 'hierarchical_dendrogram.png'.")

    def train_and_evaluate_models(self):
        print(">> Training supervised learning matrix...")

        # Classification split
        class_features = ['age', 'liked', 'disliked', 'emotional_density']
        X_c = self.cleaned_df[class_features]
        y_class = self.cleaned_df['encoded_emotion']

        # Regression split
        reg_features = ['age', 'liked', 'disliked']
        X_r = self.cleaned_df[reg_features]
        y_reg = self.cleaned_df['emotional_density']

        X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_class, test_size=0.25, random_state=42)
        X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_reg, test_size=0.25, random_state=42)

        X_train_c_scaled = self.scaler.fit_transform(X_train_c)
        X_test_c_scaled = self.scaler.transform(X_test_c)

        # Run Linear Regression baseline
        lin_reg = LinearRegression()
        lin_reg.fit(X_train_r, y_train_r)
        preds_lin = lin_reg.predict(X_test_r)
        print(f"   -> Linear Regression MSE: {mean_squared_error(y_test_r, preds_lin):.4f} | R2: {r2_score(y_test_r, preds_lin):.4f}")

        # Multi-model classification sweep
        models = {
            "Logistic Regression": LogisticRegression(max_iter=1000),
            "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
            "Naive Bayes": GaussianNB(),
            "Decision Tree": DecisionTreeClassifier(random_state=42),
            "Random Forest Ensemble": RandomForestClassifier(n_estimators=100, random_state=42)
        }

        scores = {}
        for name, clf in models.items():
            clf.fit(X_train_c_scaled, y_train_c)
            preds = clf.predict(X_test_c_scaled)
            accuracy = accuracy_score(y_test_c, preds)
            scores[name] = accuracy
            print(f"   -> {name} Accuracy: {accuracy * 100:.2f}%")

        print(f"\n* Context Note: With 9 unique target classes, a random guess baseline stands at ~11.1%.")
        print(f"  The Random Forest architecture outpaces the random baseline by {scores['Random Forest Ensemble']/0.1111:.1f}x.")
        return scores

    def execute_advanced_nlp_and_genai(self):
        """Runs Hugging Face zero-shot classification and executes the Gemini API layer."""
        print(">> Triggering cognitive inference modules...")

        # Zero-Shot classification pipeline
        try:
            nlp_classifier = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli", device=-1)
            sample_track_text = "An upbeat electronic dance track with loud bass drops and energetic rhythm."
            labels = ["joyful", "tense", "sad", "calm"]
            res = nlp_classifier(sample_track_text, candidate_labels=labels)
            print(f"   -> Hugging Face Latent Label: {res['labels'][0]} ({res['scores'][0]*100:.1f}%)")
        except Exception as e:
            print(f"   [! NLP pipeline notice]: {e}")

        # Gemini Engine Call
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            try:
                from google.colab import userdata
                api_key = userdata.get('GEMINI_API_KEY')
            except Exception:
                pass

        if not api_key:
            print("[!] Skipping GenAI block: GEMINI_API_KEY environment variable.")
            return

        try:
            client = genai.Client(api_key=api_key)
            prompt = f"""
            You are an advanced B2B AI Music Strategy Engine designed by chief data scientist {self.author}.
            We analyzed a song dataset using a GEMS evaluation mapping strategy.
            Write a 3-sentence executive business value pitch explaining how musical emotion classification models help Spotify optimize ad placement and maximize engagement for streaming platforms.
            """

            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt
            )

            print(f"\n--- Executive Insight Briefing ({self.author}) ---")
            print(response.text.strip())
            print("-" * 45)
        except Exception as e:
            print(f"   [!] Gemini API execution anomaly: {e}")


if __name__ == "__main__":
    # Instance variable named to 'runner' to ensure zero naming clashing with the imported 'pipeline' module
    runner = MusicEmotionPipeline(author="Dineshkumar")
    runner.load_kaggle_dataset()
    runner.preprocess_and_engineer_features()
    runner.cluster_audio_signatures()
    runner.train_and_evaluate_models()
    runner.execute_advanced_nlp_and_genai()

    print("\n>> Pipeline run complete. Codebase structural integrity confirmed.")

>> Ingesting data via KaggleHub...
Using Colab cache for faster access to the 'emotify-emotion-classificaiton-in-songs' dataset.
>> Dataset loaded successfully. Shape: 8407 rows, 17 columns.
>> Running preprocessing and feature engineering...
>> Preprocessing completed. Visualizations cached locally.
>> Starting unsupervised segmentation...
>> KMeans execution complete. Cluster layout:
kmeans_cluster
1    2251
0    2203
2    2173
3    1780
Name: count, dtype: int64
>> Unsupervised validation map saved as 'hierarchical_dendrogram.png'.
>> Training supervised learning matrix...
   -> Linear Regression MSE: 0.7065 | R2: 0.0530
   -> Logistic Regression Accuracy: 26.69%
   -> K-Nearest Neighbors Accuracy: 23.45%
   -> Naive Bayes Accuracy: 18.36%
   -> Decision Tree Accuracy: 27.59%
   -> Random Forest Ensemble Accuracy: 27.88%

* Context Note: With 9 unique target classes, a random guess baseline stands at ~11.1%.
  The Random Forest architecture outpaces the random baseline by 2.5x.
>> T

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

   -> Hugging Face Latent Label: joyful (82.5%)

--- Executive Insight Briefing (Dineshkumar) ---
By leveraging our advanced musical emotion classification models, streaming platforms can precisely match ad content to the listener's real-time emotional state, as derived from the music they are consuming. This granular emotional alignment significantly optimizes ad placement by ensuring relevant messages are delivered at peak receptivity, reducing listener fatigue and increasing ad recall. Ultimately, this leads to maximized user engagement, higher click-through rates for advertisers, and a direct increase in advertising revenue for the platform.
---------------------------------------------

>> Pipeline run complete. Codebase structural integrity confirmed.
